In [1]:
import wfdb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks
from scipy import signal
from sklearn.model_selection import GroupKFold


# **Load Metadata**

In [2]:
metadata = pd.read_csv('brugada-syndrome/metadata.csv')

In [3]:
print(metadata.head())
print(f"Total subjects: {len(metadata)}")
print(f"Brugada patients: {(metadata['brugada'] > 0).sum()}")
print(f"Healthy subjects: {(metadata['brugada'] == 0).sum()}")

   patient_id  basal_pattern  sudden_death  brugada
0      188981              1             0        1
1      251972              0             0        0
2      265715              0             0        0
3      267628              0             0        0
4      267630              0             0        1
Total subjects: 363
Brugada patients: 76
Healthy subjects: 287


# **Filtering**

In [4]:
patient_id = '188981'
record = wfdb.rdrecord(f'brugada-syndrome/files/{patient_id}/{patient_id}')

signals = record.p_signal
lead_names = record.sig_name
sampling_freq = record.fs

num_leads = len(lead_names)
def signal_cleaning(raw_data):
    b, a = signal.butter(3, 0.05, btype='highpass', fs=100)
    return signal.filtfilt(b, a, raw_data)

clean_signal = signal_cleaning(signals[:, 0])

# **Normalization**

In [5]:
dataset_samples = []
left_width = 40
right_width = 60
sample_id_counter = 0

In [ ]:
for index, row in metadata.iterrows():
    p_id = str(row['patient_id'])
    is_brugada = row['brugada']

    try:
        record = wfdb.rdrecord(f'brugada-syndrome/files/{p_id}/{p_id}')
        signal_raw = record.p_signal
        
        # Normalize
        norm_signals = np.zeros_like(signal_raw)

        for i in range(signal_raw.shape[1]):
            clean = signal_cleaning(signal_raw[:, i])
            if clean.max() != clean.min():
                norm_signals[:, i] = (clean - clean.min()) / (clean.max() - clean.min())
            else:
                norm_signals[:, i] = clean

        # Peak Detection
        pks, _ = find_peaks(norm_signals[:, 1], distance=50, height=0.4)

        for p in pks:
            if p > left_width and (p + right_width) < len(norm_signals):
                snip = norm_signals[p - left_width : p + right_width, :]

                dataset_samples.append({
                    'sample_id': f"SMPL_{sample_id_counter}",
                    'patient_id': p_id,
                    'label': 'Brugada' if is_brugada == 1 else 'Normal',
                    'label_code': is_brugada,
                    'signal_array': snip
                })
                sample_id_counter += 1

    except Exception as e:
        pass

print(f"{len(dataset_samples)} samples (12-Lead).")

Memulai ekstraksi 12-Lead ECG dataset...
Selesai! Terkumpul 4834 sampel (12-Lead).


# **Dataset Builder**

In [7]:
df_dataset = pd.DataFrame(dataset_samples)

In [ ]:
X_data = np.stack(df_dataset['signal_array'].values)
y_data = df_dataset['label_code'].values
groups = df_dataset['patient_id'].values

print({X_data.shape})
print("Format: (Number of Samples, Width of Cut, Number of Leads)")

Dimensi akhir X_data siap training: (4834, 100, 12)
Format: (Jumlah Sampel, Lebar Potongan, Jumlah Lead)


In [ ]:
def lihat_detak_jantung_v1(index_sample):
    if index_sample >= len(df_dataset):
        return

    data = df_dataset.iloc[index_sample]
    sinyal_12_lead = data['signal_array']

    sinyal_v1 = sinyal_12_lead[:, 6]

    print(f"DETAIL SAMPLE INDEX : {index_sample}")
    print(f"Sample ID           : {data['sample_id']}")
    print(f"Patient ID          : {data['patient_id']}")
    print(f"Diagnosis (Label)   : {data['label']} (Code: {data['label_code']})")

# **Train/Test Split**

In [9]:
gkf = GroupKFold(n_splits=5) # Using 5 splits as a common practice, but only taking one for train/test

# Perform a single split
for train_idx, test_idx in gkf.split(X_data, y_data, groups):
    X_train, X_test = X_data[train_idx], X_data[test_idx]
    y_train, y_test = y_data[train_idx], y_data[test_idx]
    # Break after the first split to get one train/test set
    break

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (3863, 100, 12)
X_test shape: (971, 100, 12)
y_train shape: (3863,)
y_test shape: (971,)


# **Handling Imbalance**

In [10]:
train_mask = y_train != 2
test_mask = y_test != 2

X_train = X_train[train_mask]
y_train = y_train[train_mask]

X_test = X_test[test_mask]
y_test = y_test[test_mask]

In [11]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight_dict = {i: w for i, w in zip(classes, class_weights)}

print(class_weight_dict)

{np.int64(0): np.float64(0.6256622516556292), np.int64(1): np.float64(2.489459815546772)}


# **Model training & evaluation**

In [12]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Build 1D CNN model for ECG classification
model = models.Sequential([
    layers.Conv1D(32, kernel_size=3, activation='relu', input_shape=(100, 12)),
    layers.MaxPooling1D(pool_size=2),

    layers.Conv1D(64, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=2),

    layers.Flatten(),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')  # Binary output
])

# Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\Helmy Krisdin Garcia\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 98, 32)         │         1,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 49, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 47, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 23, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1472)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        94,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,729 (397.38 KB)

 Trainable params: 101,729 (397.38 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# Train model with class weighting
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weight_dict,
    verbose=1
)

Epoch 1/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.5752 - loss: 0.6650 - val_accuracy: 0.6534 - val_loss: 0.6251
Epoch 2/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7297 - loss: 0.5787 - val_accuracy: 0.7831 - val_loss: 0.5947
Epoch 3/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7847 - loss: 0.4946 - val_accuracy: 0.7817 - val_loss: 0.5725
Epoch 4/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8699 - loss: 0.3467 - val_accuracy: 0.7950 - val_loss: 0.4836
Epoch 5/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8942 - loss: 0.2803 - val_accuracy: 0.7950 - val_loss: 0.5007
Epoch 6/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9172 - loss: 0.2189 - val_accuracy: 0.8135 - val_loss: 0.5167
Epoch 7/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9156 - loss: 0.2144 - val_accuracy: 0.8056 - val_loss: 0.5726
Epoch 8/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9302 - loss: 0.1705 - val_accuracy: 0.8095 - val_loss:

In [14]:
from sklearn.metrics import classification_report, confusion_matrix

# Predict on test set
y_pred = (model.predict(X_test) > 0.5).astype(int)

# Evaluation metrics
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.84      0.87       771
           1       0.50      0.65      0.57       187

    accuracy                           0.80       958
   macro avg       0.70      0.75      0.72       958
weighted avg       0.83      0.80      0.81       958

Confusion Matrix:
[[649 122]
 [ 65 122]]
